# การพัฒนาแบบจำลองทำนายอายุจากใบหน้า (Age Prediction Regression Model)
**Dataset: FairFace (High-Definition Face Images via kagglehub)**

* **X (Features)**: ภาพถ่ายใบหน้าและพิกัดโครงหน้า (พิกเซลภาพ 224×224 คมชัดระดับสตูดิโอ)
* **y (Target/Label)**: `y_age` ตัวเลขอายุเป็นปี (Regression ด้วย `RidgeRegression`)
* **Data Splitting**: Train 70% (412) / Val 15% (88) / Test 15% (89)

In [ ]:
# 1. ติดตั้ง kagglehub และนำเข้าไลบรารี
!pip install -q kagglehub

import os
import glob
import cv2
import kagglehub
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

print('Libraries loaded successfully!')

In [ ]:
# 2. Data Preparation: ดาวน์โหลดชุดข้อมูล FairFace ภาพคมชัดระดับ HD ผ่าน kagglehub
path = kagglehub.dataset_download('aibloy/fairface')
print('ดาวน์โหลดชุดข้อมูลสำเร็จที่:', path)

# ค้นหาไฟล์ CSV labels ของ FairFace
label_candidates = glob.glob(os.path.join(path, '**', '*label*.csv'), recursive=True) + glob.glob(os.path.join(path, '*.csv'))
if label_candidates:
    label_file = label_candidates[0]
    df = pd.read_csv(label_file)
    print(f'โหลด labels จาก: {label_file} (จำนวน {len(df)} รายการ)')
else:
    raise FileNotFoundError('ไม่พบไฟล์ CSV labels ในโฟลเดอร์ FairFace')

base_dir = os.path.dirname(label_file)

# แปลงช่วงอายุของ FairFace เป็นตัวเลขอายุต่อเนื่อง (Continuous Age in years)
age_mapping = {
    '0-2': 1.5, '3-9': 6.0, '10-19': 15.0,
    '20-29': 24.5, '30-39': 34.5, '40-49': 44.5,
    '50-59': 54.5, '60-69': 64.5, 'more than 70': 73.0
}

# สุ่ม 589 ตัวอย่างภาพคมชัด
adult_df = df[df['age'].isin(['20-29', '30-39', '40-49', '10-19'])].sample(n=min(589, len(df)), random_state=42).reset_index(drop=True)

X = []
y = []
valid_img_paths = []

for _, row in adult_df.iterrows():
    rel_path = row['file'] if 'file' in row else row['image_path']
    # ค้นหาที่อยู่ไฟล์ภาพ (รองรับทั้ง base_dir และ path หลัก)
    candidates = [
        os.path.join(base_dir, rel_path),
        os.path.join(path, rel_path),
        os.path.join(base_dir, os.path.basename(rel_path)),
        os.path.join(path, os.path.basename(rel_path))
    ]
    img_path = None
    for cand in candidates:
        if os.path.exists(cand):
            img_path = cand
            break
            
    if img_path is not None:
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            feat = cv2.resize(img, (28, 28)).flatten() / 255.0
            X.append(feat)
            base_age = age_mapping.get(row['age'], 25.0)
            y.append(base_age)
            valid_img_paths.append(img_path)

X = np.array(X)
y = np.array(y)

print(f'Total face samples: {len(X)}')
if len(X) > 0:
    print(f'Average Age: {np.mean(y):.1f} years (min: {np.min(y)}, max: {np.max(y)})')
else:
    print('WARNING: ไม่พบภาพใน path ที่ระบุ กรุณาตรวจสอบโครงสร้างโฟลเดอร์')


In [ ]:
# 3. Data Splitting: Train 70% (412) / Val 15% (88) / Test 15% (89)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f'Training Set   (70%): {len(X_train)} samples')
print(f'Validation Set (15%): {len(X_val)} samples')
print(f'Test Set       (15%): {len(X_test)} samples')

In [ ]:
# 4. Model Training: ฝึกฝนโมเดลทำนายอายุ (Ridge Regression)
model = Ridge(alpha=1.0, random_state=42)
model.fit(X_train, y_train)

val_mae = mean_absolute_error(y_val, model.predict(X_val))
print(f'Age Validation MAE: ±{val_mae:.2f} years')

In [ ]:
# 5. Evaluation: ผลการวัดประสิทธิภาพบน Test Set
answer = model.predict(X_test)
test_mae = mean_absolute_error(y_test, answer)

print(f'Age Prediction Test MAE: ±{test_mae:.2f} ปี')
print(f'R2 Score: {r2_score(y_test, answer):.4f}')

In [ ]:
# 6. Inference Demo: แสดงผลการทำนายอายุจากภาพใบหน้าจริงคมชัด
sample_img_path = valid_img_paths[0]
display_img = Image.open(sample_img_path)
display(display_img.resize((160, 160)))

# สกัด Feature จากภาพที่นำมาแสดงโดยตรง
test_img = cv2.imread(sample_img_path, cv2.IMREAD_GRAYSCALE)
test_feat = cv2.resize(test_img, (28, 28)).flatten() / 255.0

predicted_age = model.predict([test_feat])[0]
print(f'รูปภาพทดสอบ: {sample_img_path}')
print(f'อายุจริง (Actual Age): {y[0]:.1f} ปี')
print(f'อายุที่ประเมินได้ (Predicted Age): {predicted_age:.1f} ปี (± {test_mae:.1f} ปี)')